# Customer Churn Prediction Using Scikit-learn Pipeline API

## Objective
Build an end-to-end machine learning pipeline to predict customer churn using the Telco Churn Dataset.

## Models Used
- Logistic Regression
- Random Forest

## Techniques Used
- Pipeline API
- ColumnTransformer
- OneHotEncoder
- StandardScaler
- GridSearchCV
- Joblib Model Export


In [ ]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score, classification_report

# Load Dataset


In [ ]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

# Dataset Information

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

# Data Cleaning

In [ ]:
df.drop("customerID", axis=1, inplace=True)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].median()
)

df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

# Feature and Target Selection

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

# Identify Numerical and Categorical Features

In [ ]:
numerical_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_cols = X.select_dtypes(
    include=["object", "string"]
).columns

print(numerical_cols)
print(categorical_cols)

# Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Preprocessing Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_cols
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ]
)

# Logistic Regression Pipeline

In [ ]:
log_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

log_pipeline.fit(X_train, y_train)

log_pred = log_pipeline.predict(X_test)

print("Accuracy:",
      accuracy_score(y_test, log_pred))

print(classification_report(
    y_test,
    log_pred
))

# Random Forest Pipeline

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier",
     RandomForestClassifier(
         random_state=42
     ))
])

# Hyperparameter Tuning using GridSearchCV

In [ ]:
param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [5, 10, None]
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(
    X_train,
    y_train
)

print(grid_search.best_params_)

# Model Evaluation

In [ ]:
best_model = grid_search.best_estimator_

rf_pred = best_model.predict(X_test)

print("Accuracy:",
      accuracy_score(y_test, rf_pred))

print(classification_report(
    y_test,
    rf_pred
))

# Export Pipeline

In [ ]:
joblib.dump(
    best_model,
    "churn_pipeline.pkl"
)

print("Pipeline Saved Successfully")

# Conclusion

In this project, an end-to-end machine learning pipeline was built to predict customer churn using the Telco dataset. Data preprocessing, encoding, scaling, model training, and hyperparameter tuning were successfully implemented using Scikit-learn Pipeline API. The final model was saved using Joblib for future use.